# 07 — Feature selection

Two separate things live in `feature-selection.ipynb`, and this notebook keeps
them separate:

**Part A — the deterministic export.** `src/features/selection.select_features`,
ported from the notebook: drop `society` / `price_per_sqft`, bin `luxury_score`
and `floorNum` into `*_category`, ordinal-encode the object columns, drop the
three low-importance columns, move `price` last. Input
`gurgaon_properties_missing_value_imputation.csv` (3 554, 18) → output
`gurgaon_properties_post_feature_selection.csv` (3 554, 13). **Exact match,
13/13 columns, dtypes identical.**

**Part B — the eight feature-importance techniques.** Correlation, RF, gradient
boosting, permutation importance, LASSO, RFE, linear-regression weights, SHAP.
The original notebook runs all eight, averages five of them, and eyeballs the
result — then confirms one decision (drop `pooja room` / `study room` /
`others`) with a single 5-fold CV comparison. **None of this feeds the export.**
It is not in `src/` (deliberately). It is reproduced here, faithfully, as the
exploratory reasoning behind that one column drop — nothing more.

In [1]:
import sys, logging, warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
warnings.simplefilter("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

from src.features.selection import select_features

PROCESSED = REPO_ROOT / "data" / "processed"
mvi = pd.read_csv(PROCESSED / "gurgaon_properties_missing_value_imputation.csv")


## Part A — the deterministic export

`select_features` runs with `strict=False` here to match the notebook's
non-raising behaviour (nothing in this data falls outside the bins; see the
`strict` note below).

In [2]:
selected = select_features(mvi, strict=False)
print("input :", mvi.shape)
print("output:", selected.shape)
selected.head()


input : (3554, 18)
output: (3554, 13)


,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,price
0,0,36,3.0,2.0,2,1,850.0,0.0,0.0,0.0,1,1,0.82
1,0,95,2.0,2.0,2,1,1226.0,1.0,0.0,0.0,1,2,0.95
2,0,103,2.0,2.0,1,1,1000.0,0.0,0.0,0.0,1,0,0.32
3,0,99,3.0,4.0,4,3,1615.0,1.0,0.0,1.0,0,2,1.60
4,0,5,2.0,2.0,1,3,582.0,0.0,1.0,0.0,0,2,0.48


In [3]:
expected = pd.read_csv(PROCESSED / "gurgaon_properties_post_feature_selection.csv")
assert list(selected.columns) == list(expected.columns)
assert list(selected.dtypes.astype(str)) == list(expected.dtypes.astype(str)), "dtype mismatch"
total = match = 0
for col in expected.columns:
    a, b = selected[col], expected[col]
    m = (np.isclose(a.astype(float), b.astype(float), equal_nan=True)
         if b.dtype.kind in "fi" else a.astype(str) == b.astype(str))
    total += len(m); match += int(m.sum())
print(f"cell match vs committed file: {match}/{total} = {100 * match / total:.4f}%")
print("dtypes identical:", list(selected.dtypes.astype(str)) == list(expected.dtypes.astype(str)))


cell match vs committed file: 46202/46202 = 100.0000%
dtypes identical: True


### `strict=` — the bins have no catch-all

`_categorize_luxury` covers `luxury_score` in `[0, 175]`; `_categorize_floor`
covers integer `floorNum` in `[0, 51]`. Anything outside — a score of 176, a
floor of 52, a basement `-1` (which `cleaning._parse_floor_num` deliberately
keeps), a non-integer floor, a NaN — binned to `None` in the notebook, which
then becomes its own ordinal category. `select_features(strict=True)` (the
default) raises instead.

In [4]:
probe = mvi.copy()
probe.loc[0, "luxury_score"] = 200.0     # above the top bin
probe.loc[1, "floorNum"] = 60.0          # above the top bin

try:
    select_features(probe, strict=True)
except ValueError as e:
    print("strict=True  ->", e)

out = select_features(probe, strict=False)
print("\nstrict=False -> no error; floor_category now has an extra category:",
      sorted(out["floor_category"].unique().tolist()))


strict=True  -> 1 row(s) have a luxury_score outside the bins ([0, 50) Low, [50, 150) Medium, [150, 175] High); they would become None. Offending value(s): 200.0. Pass strict=False to reproduce the notebook's silent None.



strict=False -> no error; floor_category now has an extra category: [0, 1, 2, 3]


### Known weak spots (documented in the module)

- **Alphabetical `OrdinalEncoder` on nominal columns.** `sector` code 0 =
  `"dwarka expressway"`, `agePossession` 0 = `"Moderately Old"` — not a real
  order. The downstream model treats these as ordered ints. `_v2` (the untraced
  file) fixes some of this; `08_model_training.ipynb` re-encodes properly in its
  `ColumnTransformer`, so the export's encoding is a historical artifact.
- **The encoder is fit on the full frame**, before any train/test split.
- **`select_features(low_importance_drop=...)` is effectively pinned.** The
  function reindexes to a hard-coded 13-column `OUTPUT_COLUMNS` at the end, so
  passing anything other than the default `("pooja room", "study room",
  "others")` either no-ops or raises a `KeyError`. The parameter reads as
  configurable but only the default works — worth tightening.

## Part B — the eight feature-importance techniques (analysis only)

Rebuild the pre-drop encoded matrix the way the notebook does — this is the
same as `select_features`' internals but *before* the three columns are
dropped, so all 15 candidate features are ranked.

In [5]:
from src.features.selection import _categorize_luxury, _categorize_floor, _ordinal_encode

train = (mvi.drop(columns=["society", "price_per_sqft"])
            .assign(luxury_category=lambda d: d["luxury_score"].apply(_categorize_luxury),
                    floor_category=lambda d: d["floorNum"].apply(_categorize_floor))
            .drop(columns=["floorNum", "luxury_score"]))
encoded = _ordinal_encode(train)
X = encoded.drop(columns="price")
y = encoded["price"]
print("candidate feature matrix:", X.shape)
print("features:", list(X.columns))


candidate feature matrix: (3554, 15)
features: ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony', 'agePossession', 'built_up_area', 'study room', 'servant room', 'store room', 'pooja room', 'others', 'furnishing_type', 'luxury_category', 'floor_category']


In [6]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# 1 - correlation with price
fi_corr = encoded.corr()["price"].drop("price").rename("correlation")

# 2 - random forest impurity importance
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y)
fi_rf = pd.Series(rf.feature_importances_, index=X.columns, name="rf_importance")

# 3 - gradient boosting importance
gb = GradientBoostingRegressor(random_state=42).fit(X, y)
fi_gb = pd.Series(gb.feature_importances_, index=X.columns, name="gb_importance")

# 4 - permutation importance on a held-out split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
perm = permutation_importance(
    RandomForestRegressor(n_estimators=100, random_state=42).fit(Xtr, ytr),
    Xte, yte, n_repeats=30, random_state=42,
)
fi_perm = pd.Series(perm.importances_mean, index=X.columns, name="permutation")

pd.concat([fi_corr, fi_rf, fi_gb, fi_perm], axis=1).sort_values("rf_importance", ascending=False)


,correlation,rf_importance,gb_importance,permutation
built_up_area,0.748574,0.650726,0.677569,0.731518
sector,-0.212084,0.102406,0.102877,0.178222
property_type,0.503728,0.100067,0.098374,0.201557
bathroom,0.609777,0.026737,0.036251,0.018155
bedRoom,0.591289,0.023446,0.037718,0.022803
servant room,0.391930,0.019095,0.023195,0.021803
agePossession,-0.134171,0.014141,0.004268,0.006143
balcony,0.269637,0.012944,0.001899,0.000357
furnishing_type,0.225625,0.010156,0.002826,-0.009478
study room,0.242955,0.008770,0.003185,-0.016004


In [7]:
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
import shap

X_scaled = StandardScaler().fit_transform(X)

# 5 - LASSO coefficients
fi_lasso = pd.Series(
    Lasso(alpha=0.01, random_state=42).fit(X_scaled, y).coef_,
    index=X.columns, name="lasso_coeff",
)

# 6 - RFE (rank all features via the fitted RF)
rfe = RFE(RandomForestRegressor(random_state=42), n_features_to_select=X.shape[1], step=1).fit(X, y)
fi_rfe = pd.Series(rfe.estimator_.feature_importances_, index=X.columns, name="rfe_score")

# 7 - linear regression weights
fi_lin = pd.Series(
    LinearRegression().fit(X_scaled, y).coef_, index=X.columns, name="reg_coeff",
)

# 8 - SHAP (mean |value|) from the RF
shap_values = shap.TreeExplainer(rf).shap_values(X)
fi_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns, name="shap")

pd.concat([fi_lasso, fi_rfe, fi_lin, fi_shap], axis=1).sort_values("shap", ascending=False)


,lasso_coeff,rfe_score,reg_coeff,shap
built_up_area,1.510173,0.650726,1.512629,1.254413
property_type,0.713829,0.100067,0.712890,0.472634
sector,-0.069634,0.102406,-0.078657,0.382589
bathroom,0.275042,0.026737,0.281976,0.112964
servant room,0.160601,0.019095,0.169605,0.095413
bedRoom,0.014170,0.023446,0.016790,0.049782
balcony,-0.043562,0.012944,-0.066353,0.040373
furnishing_type,0.164113,0.010156,0.173192,0.027757
agePossession,-0.000000,0.014141,-0.002041,0.027527
floor_category,-0.002610,0.006486,-0.013482,0.024562


### The aggregate ranking

The notebook normalises each column by its sum, then averages the five
non-signed techniques (`rf`, `gb`, `permutation`, `rfe`, `shap`) — correlation
and the two linear-coefficient columns are left out of the mean because they
carry a sign.

In [8]:
fi = pd.concat([fi_rf, fi_gb, fi_perm, fi_rfe, fi_shap], axis=1)
fi_norm = fi.divide(fi.sum(axis=0), axis=1)
ranking = fi_norm.mean(axis=1).sort_values(ascending=False).rename("mean_importance")
ranking.to_frame()


,mean_importance
built_up_area,0.620011
property_type,0.131400
sector,0.122175
bathroom,0.029902
bedRoom,0.024745
servant room,0.023493
agePossession,0.009720
balcony,0.008773
store room,0.007178
furnishing_type,0.005161


### The one decision this analysis supports

Every technique puts `built_up_area` and `sector` at the top and
`pooja room` / `study room` / `others` at the bottom. The notebook's actual
test is a 5-fold CV R² with and without those three columns:

In [9]:
from sklearn.model_selection import cross_val_score

rf_cv = RandomForestRegressor(n_estimators=100, random_state=42)
all_cols = cross_val_score(rf_cv, X, y, cv=5, scoring="r2").mean()
dropped = cross_val_score(rf_cv, X.drop(columns=["pooja room", "study room", "others"]),
                          y, cv=5, scoring="r2").mean()
print(f"CV R^2 with all 15 features      : {all_cols:.4f}")
print(f"CV R^2 without the three columns : {dropped:.4f}")
print(f"difference                       : {dropped - all_cols:+.4f}")


CV R^2 with all 15 features      : 0.8203
CV R^2 without the three columns : 0.8172
difference                       : -0.0031


The difference (≈ −0.003) is well within 5-fold CV noise — dropping the three
columns neither helps nor hurts. The notebook dropped them anyway, judging
three features that sit at the bottom of every one of the eight techniques not
worth carrying. That is the entire feature-selection decision — everything
above is the supporting evidence, none of it is encoded in `select_features`.

Two asides the numbers surface:

- The absolute CV R² here (~0.82) is well below the tuned model'''s 0.91. This is
  an untuned `RandomForestRegressor(n_estimators=100)` on the **naively
  ordinal-encoded** features — `sector` as a fake-ordinal int and so on — i.e.
  the encoding smell from Part A showing up in a score.
- `floor_category` lands near the bottom, consistent with the
  `PROJECT_PLAN.md` Section 3 note that `floorNum` correlates *negatively* and
  weakly with price in this data — counter-intuitive, but low-signal across all
  eight techniques here.